In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import duckdb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)



In [2]:
# df0 = duckdb.sql("""
# SELECT *
# from read_csv_auto('C:/Users/owner/Desktop/statistical learning/IBM AML/HI-Small_Trans.csv')
# LIMIT 5
# """).df()

# display(df0)
# FROM 'HI-Small_Trans.csv'

# datapath2 = Path("C:/Users/owner/Desktop/life/10276/utility.xlsx")
# df2 = pd.read_excel(datapath2, sheet_name="Sheet1")


In [3]:
DATA_PATH = Path("C:/Users/owner/Desktop/statistical learning/IBM AML/HI-Small_Trans.csv")

# DATA_PATH = Path.cwd().parent/"HI-Small_Trans.csv" 
# DATA_PATH = Path(r"C:\Users\owner\Desktop\HI-Small_Trans.csv")
# DATA_PATH = Path.cwd().parent.parent/'statistical learning'/'IBM AML'/'HI-Small_Trans.csv'
# df=pd.read_csv("HI-Small_Trans.csv")
# print(Path.cwd())

df = pd.read_csv(DATA_PATH)       


In [4]:
print(df.info())
# print(df.shape)
# print(df.columns.tolist())
display(df.head(5))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5078345 entries, 0 to 5078344
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           object 
 1   From Bank           int64  
 2   Account             object 
 3   To Bank             int64  
 4   Account.1           object 
 5   Amount Received     float64
 6   Receiving Currency  object 
 7   Amount Paid         float64
 8   Payment Currency    object 
 9   Payment Format      object 
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 426.2+ MB
None


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [5]:
result = df.groupby("Is Laundering").size().reset_index(name="transaction_count")
result["percentage"] = (result["transaction_count"]/result["transaction_count"].sum() *100).round(4)

display(result)


,Is Laundering,transaction_count,percentage
0,0,5073168,99.8981
1,1,5177,0.1019


I analyzed the distribution of the target variable, Is Laundering, using SQL. The dataset contains approximately 5.08 million transactions, of which 5,177 are labeled as laundering transactions, representing about 0.10% of the data.
This indicates a severe class imbalance. A model that predicts every transaction as non-laundering could achieve approximately 99.90% accuracy while failing to identify any laundering transactions.
Therefore, accuracy alone is not an appropriate evaluation metric. I would focus on metrics such as recall, precision, and false-positive rate, while also considering alert volume and investigation capacity when tuning detection thresholds.

***Missing Values*** 

In [8]:
missing_count = df.isna().sum()
missing_pct = (100*missing_count / len(df)).round(4)

pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_pct
})


,missing_count,missing_percentage
Timestamp,0,0.0
From Bank,0,0.0
Account,0,0.0
To Bank,0,0.0
Account.1,0,0.0
Amount Received,0,0.0
Receiving Currency,0,0.0
Amount Paid,0,0.0
Payment Currency,0,0.0
Payment Format,0,0.0


***Duplicate row***

In [10]:
duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_count / len(df) * 100, 4), "%")


Exact duplicate rows: 9
Duplicate percentage: 0.0002 %


I checked for exact duplicate rows and found 9 duplicates out of approximately 5.08 million transactions, representing about 0.0002% of the dataset. The proportion is very small. However, I would investigate these records before removing them, because identical transaction records do not necessarily indicate data errors.

***Data Type***

In [13]:
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%Y/%m/%d %H:%M",
    errors="coerce"
)
print("New type:", df["Timestamp"].dtype)
print("Invalid timestamps:",df["Timestamp"].isna().sum())
display(df["Timestamp"].head(1))


New type: datetime64[ns]
Invalid timestamps: 0


0   2022-09-01 00:20:00
Name: Timestamp, dtype: datetime64[ns]

***Target Variable Analysis***

***1. Laundering rate by hour***

In [16]:
# Group transactions by hour
hourly_analysis = (
    df.groupby(df["Timestamp"].dt.hour)
    .agg(
        transaction_count=("Is Laundering", "size"),
        laundering_count=("Is Laundering", "sum")
    )
    .reset_index()
)

# Calculate laundering rate as a percentage
hourly_analysis["laundering_rate_pct"] = (
    100 * hourly_analysis["laundering_count"]
    / hourly_analysis["transaction_count"]
).round(4)

display(hourly_analysis)


,Timestamp,transaction_count,laundering_count,laundering_rate_pct
0,0,634726,176,0.0277
1,1,193728,152,0.0785
2,2,193143,165,0.0854
3,3,193249,146,0.0756
4,4,193166,154,0.0797
5,5,193900,188,0.0970
6,6,194456,207,0.1065
7,7,193336,195,0.1009
8,8,192921,258,0.1337
9,9,192952,217,0.1125


***2. Laundering rate by date***

In [18]:
daily_analysis = (
    df.groupby(df["Timestamp"].dt.date)
    .agg(
        transaction_count=("Is Laundering", "size"),
        laundering_count=("Is Laundering", "sum")
    )
    .reset_index()
)

daily_analysis = daily_analysis.rename(columns={"Timestamp":"Date"})

daily_analysis["laundering_rate_pct"]=(
    100 * daily_analysis["laundering_count"]
    / daily_analysis["transaction_count"]
).round(4)

print(daily_analysis)


          Date  transaction_count  laundering_count  laundering_rate_pct
0   2022-09-01            1114921               322               0.0289
1   2022-09-02             754449               408               0.0541
2   2022-09-03             207382               391               0.1885
3   2022-09-04             207430               407               0.1962
4   2022-09-05             482650               471               0.0976
5   2022-09-06             482089               531               0.1101
6   2022-09-07             482751               497               0.1030
7   2022-09-08             482773               539               0.1116
8   2022-09-09             654467               514               0.0785
9   2022-09-10             208325               442               0.2122
10  2022-09-11                396               232              58.5859
11  2022-09-12                281               170              60.4982
12  2022-09-13                184               106

In [19]:
period_summary = (
    df.groupby(
        df["Timestamp"].dt.day.apply(
            lambda d: "Sep 1-10" if d <= 10 else "Sep 11-18"
    ))
      .agg(
          transaction_count=("Is Laundering", "size"),
          laundering_count=("Is Laundering", "sum"),
          avg_amount_received=("Amount Received", "mean"),
          median_amount_received=("Amount Received", "median"),
          avg_amount_paid=("Amount Paid", "mean"),
          unique_senders=("Account", "nunique"),
          unique_receivers=("Account.1", "nunique")
      )
      .reset_index()
)

period_summary["laundering_rate_pct"] = (
    100 * period_summary["laundering_count"]
    / period_summary["transaction_count"]
).round(4)

display(period_summary)


,Timestamp,transaction_count,laundering_count,avg_amount_received,median_amount_received,avg_amount_paid,unique_senders,unique_receivers,laundering_rate_pct
0,Sep 1-10,5077237,4522,5.989319e+06,1410.220,4.509605e+06,496969,420621,0.0891
1,Sep 11-18,1108,655,3.271447e+06,13191.665,2.987995e+06,403,615,59.1155


This revealed a substantial temporal distribution shift after September 10, 2022. The transaction volume dropped from approximately 5.08 million transactions during September 1–10 to only 1,108 transactions during September 11–18. Meanwhile, the laundering label rate increased from 0.0891% to 59.1155%.
The two periods also differed substantially in transaction amount distributions and account coverage, suggesting a potential data-generation or sampling issue.
To reduce the impact of this anomaly on the initial modeling analysis, I selected transactions from **September 1–10 as the primary sample** and retained the full dataset for sensitivity analysis. This selection is provisional and does not establish that the primary sample is free from bias.

***Define primary sample***

In [22]:
df_main = df[
    (df["Timestamp"] >= "2022-09-01") &
    (df["Timestamp"] < "2022-09-11")
].copy()

# Check the sample
print("Number of transactions:", f"{len(df_main):,}")
print("Date range:",
      df_main["Timestamp"].min(),
      "to",
      df_main["Timestamp"].max())
print("Laundering transactions:", f"{df_main['Is Laundering'].sum():,}")
print("Laundering rate:",
      f"{df_main['Is Laundering'].mean() * 100:.4f}%")
display(df_main.head())


Number of transactions: 5,077,237
Date range: 2022-09-01 00:00:00 to 2022-09-10 23:59:00
Laundering transactions: 4,522
Laundering rate: 0.0891%


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


***3. Laundering rate by currency match***

In [24]:
Currency_Match = duckdb.sql("""
SELECT
    CASE
        WHEN "Receiving Currency" = "Payment Currency"
        THEN 'Same Currency'
        ELSE 'Different Currency'
    END AS currency_match,
    COUNT(*) AS transaction_count,
    SUM(
        CASE
            WHEN "Amount Received" != "Amount Paid" THEN 1
            ELSE 0
        END
    ) AS unequal_amount_count,
    sum("Is Laundering") as Laundering_count,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Amount Received" != "Amount Paid" THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS unequal_amount_pct
FROM df_main
GROUP BY currency_match
ORDER BY currency_match;
""").df()

display(Currency_Match)

,currency_match,transaction_count,unequal_amount_count,Laundering_count,unequal_amount_pct
0,Different Currency,71779,71767.0,0.0,99.98
1,Same Currency,5005458,0.0,4522.0,0.00



I explored the relationship between currency matching, amount differences, and laundering labels. I found that all same-currency transactions had equal received and paid amounts, while 99.98% of cross-currency transactions had unequal amounts. I also observed that all labeled laundering transactions appeared in the same-currency group.

***4. Laundering rate by currency***

In [27]:
result = (df_main.groupby("Receiving Currency")
    .agg(transaction_count = ("Is Laundering","size"),
    laundering_count = ("Is Laundering","sum")))

result["laundering_rate"] = (result["laundering_count"]
                             /result["transaction_count"] *100).round(4)

result = result.sort_values(by="laundering_rate",ascending=False)

display(result)


,transaction_count,laundering_count,laundering_rate
Receiving Currency,,,
Saudi Riyal,89890,302,0.3360
Euro,1171743,1211,0.1034
US Dollar,1878952,1668,0.0888
Canadian Dollar,141331,116,0.0821
Yuan,206520,169,0.0818
Australian Dollar,138476,110,0.0794
Rupee,192027,148,0.0771
Yen,156247,119,0.0762
Ruble,157330,119,0.0756


The analysis revealed variation in laundering rates across receiving currencies. Saudi Riyal had the highest observed laundering rate at approximately 0.336%, compared with the overall rate of about 0.10%. However, I would treat this as an exploratory finding rather than a conclusion about currency risk. I would investigate other factors, such as payment format, transaction amount, and transaction patterns, to determine whether the difference is associated with other characteristics of the transactions.

***5. Laundering rate by currency-specific amount***

In [30]:
amount_df = df_main.copy()

amount_df["amount_quantile"] = (
    amount_df.groupby("Receiving Currency")["Amount Received"]
    .transform(
        lambda x: pd.qcut(
            x,
            q=4,
            labels=["Q1", "Q2", "Q3", "Q4"],
            duplicates="drop"
        )
    )
)

currency_quantile_analysis = (
    amount_df.groupby(
        ["Receiving Currency", "amount_quantile"],
        observed=False
    )
    .agg(
        transaction_count=("Is Laundering", "size"),
        laundering_count=("Is Laundering", "sum"),
        median_amount=("Amount Received", "median"),
        total_amount=("Amount Received", "sum")
    )
    .reset_index()
)

currency_quantile_analysis["laundering_rate_pct"] = (
    100 * currency_quantile_analysis["laundering_count"]
    / currency_quantile_analysis["transaction_count"]
).round(4)

display(currency_quantile_analysis)


,Receiving Currency,amount_quantile,transaction_count,laundering_count,median_amount,total_amount,laundering_rate_pct
0,Australian Dollar,Q1,34639,7,6.987000e+01,2.974929e+06,0.0202
1,Australian Dollar,Q2,34602,16,5.509600e+02,2.149674e+07,0.0462
2,Australian Dollar,Q3,34622,46,3.129660e+03,1.232981e+08,0.1329
3,Australian Dollar,Q4,34613,41,3.275269e+04,4.717678e+10,0.1185
4,Bitcoin,Q1,37047,4,2.388000e-03,1.286751e+02,0.0108
5,Bitcoin,Q2,37035,7,2.609300e-02,1.114653e+03,0.0189
6,Bitcoin,Q3,37034,34,1.582600e-01,6.747402e+03,0.0918
7,Bitcoin,Q4,37035,11,1.878751e+00,3.031543e+06,0.0297
8,Brazil Real,Q1,17884,4,2.809300e+02,6.190747e+06,0.0224
9,Brazil Real,Q2,17889,6,2.233330e+03,4.554377e+07,0.0335


I analyzed transaction amounts using currency-specific quartiles. For most currencies, higher transaction amount quartiles showed higher proportions of positive-labeled transactions. However, the pattern was not strictly monotonic across all currencies. This suggests that transaction amount may provide useful predictive information, but I would validate its contribution through model testing.

***6. Laundering count by payment format***

In [33]:
duckdb.sql("""
select "Payment Format",
count(*) as transaction_count,
sum("Is Laundering") as Laundering_count,
round(100*sum("Is Laundering")/count(*),4) as Laundering_rate
from df_main
group by "Payment Format";
""")


┌────────────────┬───────────────────┬──────────────────┬─────────────────┐
│ Payment Format │ transaction_count │ Laundering_count │ Laundering_rate │
│    varchar     │       int64       │      int128      │     double      │
├────────────────┼───────────────────┼──────────────────┼─────────────────┤
│ Bitcoin        │            146091 │               56 │          0.0383 │
│ Cheque         │           1864331 │              324 │          0.0174 │
│ Credit Card    │           1323324 │              206 │          0.0156 │
│ Wire           │            171855 │                0 │             0.0 │
│ Reinvestment   │            481056 │                0 │             0.0 │
│ ACH            │            599689 │             3828 │          0.6383 │
│ Cash           │            490891 │              108 │           0.022 │
└────────────────┴───────────────────┴──────────────────┴─────────────────┘

In [34]:
total_laundering = df_main["Is Laundering"].sum()

ach_laundering = df_main.loc[df_main["Payment Format"] == "ACH","Is Laundering"].sum()

ach_positive_share = (ach_laundering / total_laundering * 100)

print("Total laundering transactions:", total_laundering)
print("ACH laundering transactions:", ach_laundering)
print("ACH share of all laundering cases:",
      round(ach_positive_share, 2), "%")


Total laundering transactions: 4522
ACH laundering transactions: 3828
ACH share of all laundering cases: 84.65 %


The analysis revealed a strong association between payment format and laundering labels in the primary sample.
ACH had the highest laundering rate at 0.6383%, with 3,828 laundering-labeled transactions, accounting for 84.65% of all positive cases in the dataset. Cash had a laundering rate of 0.0220%, followed by Bitcoin at 0.0383%, Cheque at 0.0174%, and Credit Card at 0.0156%. No laundering-labeled transactions were observed for Reinvestment or Wire in this dataset.
These findings suggest that Payment format may help distinguish suspicious transactions because laundering rates vary across payment methods in this dataset. However, I would validate this pattern before using it in a risk model. Since the dataset is highly imbalanced, a payment method with no positive labels in this sample does not necessarily mean it carries no risk in practice.

***7. Laundering rate by sender account***

In [37]:
res = df_main.groupby("Account").agg(
    transaction_count=("Is Laundering", "size"),
    laundering_count=("Is Laundering", "sum")
)

res["laundering_rate_pct"] = (res["laundering_count"] / res["transaction_count"] * 100).round(4)
res_sorted = res.sort_values(by="laundering_count", ascending=False)

display(res_sorted.head(20))


,transaction_count,laundering_count,laundering_rate_pct
Account,,,
100428660,168672,243,0.1441
1004286A8,103018,158,0.1534
100428978,20497,29,0.1415
100428810,16426,26,0.1583
80266F880,95,26,27.3684
812D22980,25,23,92.0000
100428738,13756,23,0.1672
100428780,17264,21,0.1216
1004286F0,18663,21,0.1125


In [38]:
res_sorted = res.sort_values(by="laundering_count", ascending=False).copy()
total_laundering = res_sorted["laundering_count"].sum()

top_counts = tuple((10, 50, 100, 500))
for n in top_counts:
    top_share = (res_sorted.head(n)["laundering_count"].sum() / total_laundering * 100)
    print(f"Top {n} sender accounts account for {top_share:.2f}% of all laundering.")
print("Total",sum(df_main["Is Laundering"]), "laundering accounts" )




Top 10 sender accounts account for 13.05% of all laundering.
Top 50 sender accounts account for 25.32% of all laundering.
Top 100 sender accounts account for 32.88% of all laundering.
Top 500 sender accounts account for 43.92% of all laundering.
Total 4522 laundering accounts


In [39]:
res_sorted["transaction_volume_group"] = pd.cut(
    res_sorted["transaction_count"],
    bins=[0, 10, 50, 100, 1000, 10000, float("inf")],
    labels=["1-10", "11-50", "51-100", "101-1K", "1K-10K", "10K+"]
)

volume_analysis = (
    res_sorted.groupby("transaction_volume_group", observed=False)
    .agg(
        account_count=("transaction_count", "size"),
        transaction_count=("transaction_count", "sum"),
        laundering_count=("laundering_count", "sum")
    )
    .reset_index()
)

volume_analysis["laundering_rate_pct"] = (
    volume_analysis["laundering_count"] / volume_analysis["transaction_count"] * 100
).round(4)

display(volume_analysis)


,transaction_volume_group,account_count,transaction_count,laundering_count,laundering_rate_pct
0,1-10,393868,844724,2130,0.2522
1,11-50,81554,2089823,1329,0.0636
2,51-100,17803,1211281,287,0.0237
3,101-1K,3729,481550,143,0.0297
4,1K-10K,3,24192,35,0.1447
5,10K+,12,425667,598,0.1405


***8. Laundering rate by receiver account***

In [41]:
res1 = df_main.groupby("Account.1").agg(
    transaction_count=("Is Laundering", "size"),
    laundering_count=("Is Laundering", "sum")
)

res1["laundering_rate_pct"] = (res1["laundering_count"] / res1["transaction_count"] * 100).round(4)
res1_sorted = res1.sort_values(by="laundering_count", ascending=False)

display(res1_sorted.head(20))


,transaction_count,laundering_count,laundering_rate_pct
Account.1,,,
811C599A0,52,26,50.0000
8021353D0,49,25,51.0204
80266F880,74,21,28.3784
811C597B0,48,21,43.7500
811ED7DF0,56,19,33.9286
812A09CF0,19,18,94.7368
80AEBECB0,27,16,59.2593
811D80C30,29,16,55.1724
8028F2650,18,16,88.8889


In [42]:
res1_sorted = res1.sort_values(by="laundering_count", ascending=False).copy()
total_laundering_receiver = res1_sorted["laundering_count"].sum()

top_counts_rec = tuple((10, 50, 100, 500))
for n in top_counts_rec:
    top_share_rec = (res1_sorted.head(n)["laundering_count"].sum() / total_laundering_receiver * 100)
    print(f"Top {n} receiver accounts account for {top_share_rec:.2f}% of all laundering.")
print("Total", sum(df_main["Is Laundering"]),"laundering accounts")



Top 10 receiver accounts account for 4.29% of all laundering.
Top 50 receiver accounts account for 15.04% of all laundering.
Top 100 receiver accounts account for 20.52% of all laundering.
Top 500 receiver accounts account for 30.98% of all laundering.
Total 4522 laundering accounts


In [43]:
res1_sorted["transaction_volume_group"] = pd.cut(
    res1_sorted["transaction_count"],
    bins=[0, 10, 50, 100, 1000, 10000, float("inf")],
    labels=["1-10", "11-50", "51-100", "101-1K", "1K-10K", "10K+"]
)

volume1_analysis = (
    res1_sorted.groupby("transaction_volume_group", observed=False)
    .agg(
        account_count=("transaction_count", "size"),
        transaction_count=("transaction_count", "sum"),
        laundering_count=("laundering_count", "sum")
    )
    .reset_index()
)

volume1_analysis["laundering_rate_pct"] = (
    volume1_analysis["laundering_count"] / volume1_analysis["transaction_count"] * 100
).round(4)

display(volume1_analysis)


,transaction_volume_group,account_count,transaction_count,laundering_count,laundering_rate_pct
0,1-10,245773,596020,1574,0.2641
1,11-50,170247,4213255,2746,0.0652
2,51-100,4587,264773,201,0.0759
3,101-1K,13,2105,1,0.0475
4,1K-10K,1,1084,0,0.0000
5,10K+,0,0,0,NaN


I conducted account-level analysis from both sender and receiver perspectives to examine the concentration and distribution of laundering-labeled transactions.
Account Concentration: The concentration analysis showed that laundering-labeled transactions were more concentrated among sender accounts than receiver accounts. The top 10, 50, 100, and 500 sender accounts accounted for 13.05%, 25.32%, 32.88%, and 43.92% of all laundering-labeled transactions, respectively. The corresponding receiver account groups accounted for 4.29%, 15.04%, 20.52%, and 30.98%.
Transaction Volume Analysis: I also grouped sender and receiver accounts by their transaction volumes. Both sides showed relatively high laundering-labeled transaction rates among accounts with 1–10 transactions (0.2521% for senders and 0.2641% for receivers). For accounts with 51–100 transactions, the receiver-side rate was 0.0759%, compared with 0.0237% on the sender side.

***9. Laundering rate by Counterparty Diversity***

In [46]:
sender_counterparty = duckdb.sql("""
SELECT
    "Account" AS sender_account,
    COUNT(*) AS transaction_count,
    COUNT(DISTINCT "Account.1") AS unique_receivers,
    SUM("Is Laundering") AS laundering_count,
    MIN("Amount Paid") FILTER (WHERE "Is Laundering" = 1) AS laundering_amount_min,
    MAX("Amount Paid") FILTER (WHERE "Is Laundering" = 1) AS laundering_amount_max,
    AVG("Amount Paid") FILTER (WHERE "Is Laundering" = 1) AS laundering_amount_mean,
    AVG("Amount Paid") AS all_amount_mean
FROM df_main
GROUP BY "Account"
""").df()

sender_counterparty["laundering_pct"] = (
    sender_counterparty["laundering_count"] / sender_counterparty["transaction_count"] * 100
).round(4)

sender_counterparty = sender_counterparty.sort_values(by="laundering_count", ascending=False)
display(sender_counterparty.head(10))


,sender_account,transaction_count,unique_receivers,laundering_count,laundering_amount_min,laundering_amount_max,laundering_amount_mean,all_amount_mean,laundering_pct
116358,100428660,168672,14230,243.0,2.28,20030000.28,2.059618e+05,3.128100e+05,0.1441
342179,1004286A8,103018,8846,158.0,0.33,15428581.40,2.024870e+05,2.530565e+05,0.1534
148525,100428978,20497,1776,29.0,14.53,71232.71,1.083762e+04,3.674738e+05,0.1415
62585,80266F880,95,5,26.0,25.22,17049.78,7.815757e+03,3.291640e+04,27.3684
55744,100428810,16426,1394,26.0,7.80,9857.57,1.784747e+03,3.137475e+05,0.1583
79218,812D22980,25,4,23.0,655.64,74116.00,4.158180e+04,3.830505e+04,92.0000
474972,100428738,13756,1136,23.0,3114.09,37080124.05,1.982012e+06,1.884391e+07,0.1672
467259,100428780,17264,1421,21.0,152.58,1225855.08,1.663010e+05,2.165974e+07,0.1216
366421,1004286F0,18663,1575,21.0,229.82,170123.56,3.592955e+04,1.321189e+06,0.1125
178769,8021353D0,21,2,20.0,946.84,17169.48,8.638613e+03,8.228260e+03,95.2381


In [47]:
duckdb.register('sender_counterparty', sender_counterparty)

sender_counterparty_analysis = duckdb.sql("""
    SELECT
        CASE
            WHEN unique_receivers = 1 THEN '1'
            WHEN unique_receivers BETWEEN 2 AND 5 THEN '2-5'
            WHEN unique_receivers BETWEEN 6 AND 10 THEN '6-10'
            WHEN unique_receivers BETWEEN 11 AND 20 THEN '11-20'
            WHEN unique_receivers BETWEEN 21 AND 50 THEN '21-50'
            WHEN unique_receivers BETWEEN 51 AND 100 THEN '51-100'
            WHEN unique_receivers BETWEEN 101 AND 200 THEN '101-200'
            ELSE '200+'
        END AS counterparty_range,
        COUNT(*) AS account_count,
        SUM(transaction_count) AS transaction_count,
        SUM(laundering_count) AS laundering_count,
        ROUND(100.0 * SUM(laundering_count) / SUM(transaction_count), 4) AS laundering_rate_pct
    FROM sender_counterparty
    GROUP BY counterparty_range
    ORDER BY MIN(unique_receivers)
""").df()

display(sender_counterparty_analysis)


,counterparty_range,account_count,transaction_count,laundering_count,laundering_rate_pct
0,1,287327,507026.0,139.0,0.0274
1,2-5,184706,2780652.0,2810.0,0.1011
2,6-10,20930,1025878.0,414.0,0.0404
3,11-20,3859,293726.0,501.0,0.1706
4,21-50,132,20096.0,25.0,0.1244
5,200+,15,449859.0,633.0,0.1407


In [48]:
receiver_counterparty = duckdb.sql("""
    SELECT 
        "Account.1" as receiver_account,
        COUNT(*) as transaction_count,
        COUNT(DISTINCT "Account") as unique_senders,
        SUM("Is Laundering") as laundering_count,
        ROUND(SUM("Is Laundering")*100/COUNT(*), 4) as laundering_rate
    FROM df_main
    GROUP BY "Account.1"
    ORDER BY laundering_count DESC;
""").df()

display(receiver_counterparty.head(20))


,receiver_account,transaction_count,unique_senders,laundering_count,laundering_rate
0,811C599A0,52,7,26.0,50.0000
1,8021353D0,49,4,25.0,51.0204
2,811C597B0,48,6,21.0,43.7500
3,80266F880,74,4,21.0,28.3784
4,811ED7DF0,56,20,19.0,33.9286
5,812A09CF0,19,2,18.0,94.7368
6,8028F2650,18,18,16.0,88.8889
7,811D80C30,29,17,16.0,55.1724
8,80AEBECB0,27,19,16.0,59.2593
9,809CC4600,45,21,16.0,35.5556


In [49]:
duckdb.register('receiver_counterparty', receiver_counterparty)

receiver_counterparty_analysis = duckdb.sql("""
    SELECT
        CASE
            WHEN unique_senders = 1 THEN '1'
            WHEN unique_senders BETWEEN 2 AND 5 THEN '2-5'
            WHEN unique_senders BETWEEN 6 AND 10 THEN '6-10'
            WHEN unique_senders BETWEEN 11 AND 20 THEN '11-20'
            WHEN unique_senders BETWEEN 21 AND 50 THEN '21-50'
            WHEN unique_senders BETWEEN 51 AND 100 THEN '51-100'
            WHEN unique_senders BETWEEN 101 AND 200 THEN '101-200'
            ELSE '200+'
        END AS counterparty_range,
        COUNT(*) AS account_count,
        SUM(transaction_count) AS transaction_count,
        SUM(laundering_count) AS laundering_count,
        ROUND(100.0 * SUM(laundering_count) / SUM(transaction_count), 4) AS laundering_rate_pct
    FROM receiver_counterparty
    GROUP BY counterparty_range
    ORDER BY MIN(unique_senders)
""").df()

display(receiver_counterparty_analysis)


,counterparty_range,account_count,transaction_count,laundering_count,laundering_rate_pct
0,1,161881,282931.0,65.0,0.0230
1,2-5,246223,4473938.0,3091.0,0.0691
2,6-10,9665,239970.0,625.0,0.2604
3,11-20,1812,36345.0,638.0,1.7554
4,21-50,773,24497.0,83.0,0.3388
5,51-100,265,17819.0,20.0,0.1122
6,200+,2,1737.0,0.0,0.0000


I analyzed sender and receiver account activities based on the number of unique counterparty relationships during the observation period.
The results showed a non-linear relationship between counterparty diversity and the proportion of laundering-labeled transactions. For both senders and receivers, the 2–5 counterparty group accounted for the absolute largest number of laundering-labeled transactions (62.1% share on the sender side and 68.4% share on the receiver side), primarily driven by its massive baseline transaction volume.
Compared with sender-side analysis, receiver accounts showed a more pronounced spike in laundering rates within the 11–20 unique counterparties range (1.7554%). This indicates that receiver-side interaction diversity may serve as a highly effective behavioral indicator for structural anomalies in this transaction network.

***Feature Engineering***

***1. Extracting historical behavioral features via Window Functions***

In [53]:
df_main_features = duckdb.sql("""
WITH

-- 1. Count transactions per sender account at each specific timestamp
sender_tx_by_time AS (
    SELECT
        "Account" AS sender_account,
        "Timestamp" AS ts,
        COUNT(*) AS tx_count
    FROM df_main
    GROUP BY "Account", "Timestamp"
),

-- 2. Calculate cumulative historical transaction count for senders (prior to current timestamp)
sender_tx_history AS (
    SELECT
        sender_account,
        ts,
        COALESCE(
            SUM(tx_count) OVER (
                PARTITION BY sender_account
                ORDER BY ts
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0
        ) AS sender_transaction_count
    FROM sender_tx_by_time
),

-- 3. Identify the first transaction timestamp between each sender and receiver pair
sender_receiver_first AS (
    SELECT
        "Account" AS sender_account,
        "Account.1" AS receiver_account,
        MIN("Timestamp") AS first_ts
    FROM df_main
    GROUP BY "Account", "Account.1"
),

-- 4. Count new unique receivers added by each sender at each timestamp
sender_new_receivers AS (
    SELECT
        sender_account,
        first_ts AS ts,
        COUNT(*) AS new_receivers
    FROM sender_receiver_first
    GROUP BY sender_account, first_ts
),

-- 5. Calculate cumulative historical unique receivers contacted by senders (prior to current timestamp)
sender_unique_history AS (
    SELECT
        sender_account,
        ts,
        COALESCE(
            SUM(new_receivers) OVER (
                PARTITION BY sender_account
                ORDER BY ts
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0
        ) AS sender_unique_receivers
    FROM sender_new_receivers
),

-- 6. Count transactions per receiver account at each specific timestamp
receiver_tx_by_time AS (
    SELECT
        "Account.1" AS receiver_account,
        "Timestamp" AS ts,
        COUNT(*) AS tx_count
    FROM df_main
    GROUP BY "Account.1", "Timestamp"
),

-- 7. Calculate cumulative historical transaction count for receivers (prior to current timestamp)
receiver_tx_history AS (
    SELECT
        receiver_account,
        ts,
        COALESCE(
            SUM(tx_count) OVER (
                PARTITION BY receiver_account
                ORDER BY ts
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0
        ) AS receiver_transaction_count
    FROM receiver_tx_by_time
),

-- 8. Identify the first transaction timestamp between each receiver and sender pair
receiver_sender_first AS (
    SELECT
        "Account.1" AS receiver_account,
        "Account" AS sender_account,
        MIN("Timestamp") AS first_ts
    FROM df_main
    GROUP BY "Account.1", "Account"
),

-- 9. Count new unique senders added by each receiver at each timestamp
receiver_new_senders AS (
    SELECT
        receiver_account,
        first_ts AS ts,
        COUNT(*) AS new_senders
    FROM receiver_sender_first
    GROUP BY receiver_account, first_ts
),

-- 10. Calculate cumulative historical unique senders associated with receivers (prior to current timestamp)
receiver_unique_history AS (
    SELECT
        receiver_account,
        ts,
        COALESCE(
            SUM(new_senders) OVER (
                PARTITION BY receiver_account
                ORDER BY ts
                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
            ), 0
        ) AS receiver_unique_senders
    FROM receiver_new_senders
)

-- 11. Merge all four baseline behavioral features back into the transaction-level dataset
SELECT
    d.*,
    CAST(COALESCE(st.sender_transaction_count, 0) AS BIGINT) AS sender_transaction_count,
    CAST(COALESCE(su.sender_unique_receivers, 0) AS BIGINT) AS sender_unique_receivers,
    CAST(COALESCE(rt.receiver_transaction_count, 0) AS BIGINT) AS receiver_transaction_count,
    CAST(COALESCE(ru.receiver_unique_senders, 0) AS BIGINT) AS receiver_unique_senders
FROM df_main d
LEFT JOIN sender_tx_history st ON d."Account" = st.sender_account AND d."Timestamp" = st.ts
LEFT JOIN sender_unique_history su ON d."Account" = su.sender_account AND d."Timestamp" = su.ts
LEFT JOIN receiver_tx_history rt ON d."Account.1" = rt.receiver_account AND d."Timestamp" = rt.ts
LEFT JOIN receiver_unique_history ru ON d."Account.1" = ru.receiver_account AND d."Timestamp" = ru.ts
""").df()

print("Engineered dataset shape:", df_main_features.shape)
display(df_main_features.head(20))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Engineered dataset shape: (5077237, 15)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,sender_transaction_count,sender_unique_receivers,receiver_transaction_count,receiver_unique_senders
0,2022-09-01 00:16:00,222,811B6D940,222,811B6D940,25.70,Saudi Riyal,25.70,Saudi Riyal,Reinvestment,0,2,1,0,0
1,2022-09-01 00:23:00,118,811B6E170,118,811B6E170,86.34,Saudi Riyal,86.34,Saudi Riyal,Reinvestment,0,0,0,0,0
2,2022-09-01 00:01:00,119,811B6E2B0,119,811B6E2B0,25.60,Saudi Riyal,25.60,Saudi Riyal,Reinvestment,0,0,0,0,0
3,2022-09-01 00:26:00,222,811B6EEF0,222,811B6EEF0,219023.79,Saudi Riyal,219023.79,Saudi Riyal,Reinvestment,0,0,0,0,0
4,2022-09-01 00:23:00,118,811B6EF40,118,811B6EF40,35561.52,Saudi Riyal,35561.52,Saudi Riyal,Reinvestment,0,0,0,0,0
5,2022-09-01 00:18:00,119,811B6FA50,119,811B6FA50,46.62,Saudi Riyal,46.62,Saudi Riyal,Reinvestment,0,0,0,0,0
6,2022-09-01 00:00:00,223,811B70F10,223,811B70F10,70.29,Saudi Riyal,70.29,Saudi Riyal,Reinvestment,0,0,0,0,0
7,2022-09-01 00:27:00,223,811B716A0,223,811B716A0,17.40,Saudi Riyal,17.40,Saudi Riyal,Reinvestment,0,0,0,0,0
8,2022-09-01 00:24:00,222,811B71D00,222,811B71D00,50.30,Saudi Riyal,50.30,Saudi Riyal,Reinvestment,0,0,0,0,0
9,2022-09-01 00:09:00,223,811B71430,223,811B71430,1927.25,Saudi Riyal,1927.25,Saudi Riyal,Reinvestment,0,0,0,0,0


In [54]:
# new_features = [
#     "sender_transaction_count",
#     "sender_unique_receivers",
#     "receiver_transaction_count",
#     "receiver_unique_senders"
# ]

# print("Original table shape:", df_main.shape)
# print("New feature table shape:", df_main_features.shape)

# # Check for missing values in the newly engineered columns
# print("\nMissing values status:")
# display(df_main_features[new_features].isnull().sum())

# # Generate descriptive statistics grouped by target
# feature_comparison = (
#     df_main_features.groupby("Is Laundering")[new_features]
#     .agg(["mean", "median"])
# )
# display(feature_comparison.style.format("{:,.1f}"))


***2. Advanced rolling velocity analysis within a specific horizon***

In [56]:
frequency_analysis_2m = duckdb.sql("""
WITH sender_frequency AS (
    SELECT
        "Is Laundering",
        COUNT(*) OVER (
            PARTITION BY "Account"
            ORDER BY EPOCH("Timestamp")
            RANGE BETWEEN 120 PRECEDING AND 1 PRECEDING
        ) AS sender_2m_frequency
    FROM df_main
)
SELECT
    CASE
        WHEN sender_2m_frequency = 0 THEN '0'
        WHEN sender_2m_frequency = 1 THEN '1'
        WHEN sender_2m_frequency BETWEEN 2 AND 3 THEN '2-3'
        WHEN sender_2m_frequency BETWEEN 4 AND 5 THEN '4-5'
        WHEN sender_2m_frequency BETWEEN 6 AND 10 THEN '6-10'
        WHEN sender_2m_frequency BETWEEN 11 AND 15 THEN '11-15'
        WHEN sender_2m_frequency BETWEEN 16 AND 20 THEN '16-20'
        WHEN sender_2m_frequency BETWEEN 21 AND 25 THEN '21-25'
        WHEN sender_2m_frequency BETWEEN 26 AND 30 THEN '26-30'
        ELSE '30+'
    END AS frequency_group,
    COUNT(*) AS transaction_count,
    SUM("Is Laundering") AS laundering_count,
    ROUND(
        100.0 * SUM("Is Laundering") / COUNT(*), 4
    ) AS laundering_rate_pct
FROM sender_frequency
GROUP BY frequency_group
ORDER BY CASE frequency_group
    WHEN '0' THEN 1
    WHEN '1' THEN 2
    WHEN '2-3' THEN 3
    WHEN '4-5' THEN 4
    WHEN '6-10' THEN 5
    WHEN '11-15' THEN 6
    WHEN '16-20' THEN 7
    WHEN '21-25' THEN 8
    WHEN '26-30' THEN 9
    ELSE 10
END
""").df()

with pd.option_context("display.float_format", "{:.4f}".format):
    display(frequency_analysis_2m)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,frequency_group,transaction_count,laundering_count,laundering_rate_pct
0,0,4288681,3918.0000,0.0914
1,1,374888,67.0000,0.0179
2,2-3,97445,81.0000,0.0831
3,4-5,37271,43.0000,0.1154
4,6-10,35321,56.0000,0.1585
5,11-15,36905,49.0000,0.1328
6,16-20,46689,72.0000,0.1542
7,21-25,50896,75.0000,0.1474
8,26-30,47895,74.0000,0.1545
9,30+,61246,87.0000,0.1421


In [90]:
# Create an explicit frequency view using duckdb
frequency_analysis_1h = duckdb.sql("""
WITH sender_frequency AS (
    SELECT
        "Is Laundering",
        COUNT(*) OVER (
            PARTITION BY "Account"
            ORDER BY EPOCH("Timestamp")
            RANGE BETWEEN 3600 PRECEDING AND 1 PRECEDING
        ) AS sender_1h_frequency
    FROM df_main
)
SELECT
    CASE
        WHEN sender_1h_frequency = 0 THEN '0'
        WHEN sender_1h_frequency = 1 THEN '1'
        WHEN sender_1h_frequency BETWEEN 2 AND 3 THEN '2-3'
        WHEN sender_1h_frequency BETWEEN 4 AND 5 THEN '4-5'
        WHEN sender_1h_frequency BETWEEN 6 AND 10 THEN '6-10'
        WHEN sender_1h_frequency BETWEEN 11 AND 15 THEN '11-15'
        WHEN sender_1h_frequency BETWEEN 16 AND 20 THEN '16-20'
        ELSE '20+'
    END AS frequency_group,

    COUNT(*) AS transaction_count,
    SUM("Is Laundering") AS laundering_count,

    ROUND(
        100.0 * SUM("Is Laundering") / COUNT(*), 4
    ) AS laundering_rate_pct

FROM sender_frequency
GROUP BY frequency_group
ORDER BY
    CASE frequency_group
        WHEN '0' THEN 1
        WHEN '1' THEN 2
        WHEN '2-3' THEN 3
        WHEN '4-5' THEN 4
        WHEN '6-10' THEN 5
        WHEN '11-15' THEN 6
        WHEN '16-20' THEN 7
        ELSE 8
    END
""").df()

with pd.option_context("display.float_format", "{:.4f}".format):
    display(frequency_analysis_1h)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,frequency_group,transaction_count,laundering_count,laundering_rate_pct
0,0,2482573,3658.0000,0.1473
1,1,1316571,85.0000,0.0065
2,2-3,707726,132.0000,0.0187
3,4-5,101185,12.0000,0.0119
4,6-10,20314,3.0000,0.0148
5,11-15,1988,1.0000,0.0503
6,16-20,3246,3.0000,0.0924
7,20+,443634,628.0000,0.1416


In [94]:
amount_analysis_1h = duckdb.sql("""
WITH sender_amount AS (
    SELECT
        "Is Laundering",
        SUM("Amount Paid") OVER (
            PARTITION BY "Account"
            ORDER BY EPOCH("Timestamp")
            RANGE BETWEEN 3600 PRECEDING AND 1 PRECEDING
        ) AS sender_1h_total_paid
    FROM df_main
)

SELECT
    CASE
        WHEN sender_1h_total_paid = 0 THEN '0'
        WHEN sender_1h_total_paid <= 1000 THEN '0-1K'
        WHEN sender_1h_total_paid <= 10000 THEN '1K-10K'
        WHEN sender_1h_total_paid <= 100000 THEN '10K-100K'
        WHEN sender_1h_total_paid <= 1000000 THEN '100K-1M'
        WHEN sender_1h_total_paid <= 10000000 THEN '1M-10M'
        ELSE '10M+'
    END AS amount_group,

    COUNT(*) AS transaction_count,
    SUM("Is Laundering") AS laundering_count,

    ROUND(
        100.0 * SUM("Is Laundering") / COUNT(*), 4
    ) AS laundering_rate_pct

FROM sender_amount
GROUP BY amount_group
ORDER BY
    CASE amount_group
        WHEN '0' THEN 1
        WHEN '0-1K' THEN 2
        WHEN '1K-10K' THEN 3
        WHEN '10K-100K' THEN 4
        WHEN '100K-1M' THEN 5
        WHEN '1M-10M' THEN 6
        ELSE 7
    END
""").df()

print(amount_analysis_1h.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

amount_group  transaction_count  laundering_count  laundering_rate_pct
        0-1K             817671              65.0               0.0079
      1K-10K             601648              62.0               0.0103
    10K-100K             396519              81.0               0.0204
     100K-1M             226846              41.0               0.0181
      1M-10M             145669              87.0               0.0597
        10M+            2888884            4186.0               0.1449


Analyzed sender transaction velocity across multiple time windows and observed a higher laundering-label rate among accounts with more than 10 transactions in the preceding hour. High historical payment volume within a one-hour window (the 10M+ total paid group) is also strongly associated with an increased laundering rate in this dataset, identifying short-term transaction velocity as a highly effective candidate feature for transaction monitoring.

In [96]:
# 1. Compute rolling 1-hour window statistics (frequency and total amount paid)
df_main_features = duckdb.sql("""
SELECT
    *,
    COUNT(*) OVER (
        PARTITION BY "Account"
        ORDER BY EPOCH("Timestamp")
        RANGE BETWEEN 3600 PRECEDING AND 1 PRECEDING
    ) AS sender_1h_frequency,

    SUM("Amount Paid") OVER (
        PARTITION BY "Account"
        ORDER BY EPOCH("Timestamp")
        RANGE BETWEEN 3600 PRECEDING AND 1 PRECEDING
    ) AS sender_1h_total_paid
FROM df_main_features
""").df()

# 2. Final pipeline validation
display(
    df_main_features[
        [
            "Account",
            "Timestamp",
            "Is Laundering",
            "sender_1h_frequency",
            "sender_1h_total_paid"
        ]
    ].head(5)
)

print("Final engineered dataset row count:", len(df_main_features))
print("Missing values in 1h frequency:", df_main_features["sender_1h_frequency"].isna().sum())
print("Data type of 1h frequency:", df_main_features["sender_1h_frequency"].dtype)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Account,Timestamp,Is Laundering,sender_1h_frequency,sender_1h_total_paid
0,100428660,2022-09-01,0,0,NaN
1,100428660,2022-09-01,0,0,NaN
2,100428660,2022-09-01,0,0,NaN
3,100428660,2022-09-01,0,0,NaN
4,100428660,2022-09-01,0,0,NaN


Final engineered dataset row count: 5077237
Missing values in 1h frequency: 0
Data type of 1h frequency: int64


In [98]:
df_main_features["sender_1h_total_paid"] = (
    df_main_features["sender_1h_total_paid"].fillna(0)
)

print("Missing values after treatment:")
print(df_main_features["sender_1h_total_paid"].isna().sum())

Missing values after treatment:
0


***3. log transformation to transaction amount***
 
 Apply log transformation to reduce right skewness and minimize the influence of extreme transaction amounts.

In [100]:
df_main_features["log_amount_received"] = np.log1p(
    df_main_features["Amount Received"]
)

***4. Extracting hour, date, day of the week as features***

In [59]:
df_main_features["transaction_hour"] = (
    df_main_features["Timestamp"].dt.hour)

df_main_features["transaction_day"] = (
    df_main_features["Timestamp"].dt.day)

df_main_features["transaction_weekday"] = (
    df_main_features["Timestamp"].dt.dayofweek)

***Determine training set and test set***

In [60]:
train = (
    df_main_features["Timestamp"] < "2022-09-09"
)

test = (
    (df_main_features["Timestamp"] >= "2022-09-09") &
    (df_main_features["Timestamp"] < "2022-09-11")
)

print("Training transactions:", train.sum())
print("Testing transactions:", test.sum())

print(
    "Training laundering:",
    df_main_features.loc[train, "Is Laundering"].sum()
)

print(
    "Testing laundering:",
    df_main_features.loc[test, "Is Laundering"].sum()
)

Training transactions: 4214445
Testing transactions: 862792
Training laundering: 3566
Testing laundering: 956


In [61]:
print("Training laundering rate:",
      df_main_features.loc[train, "Is Laundering"].mean() * 100)

print("Testing laundering rate:",
      df_main_features.loc[test, "Is Laundering"].mean() * 100)

Training laundering rate: 0.08461375103957935
Testing laundering rate: 0.11080306725143488


***Determine Predictors***

In [62]:
numeric_features = [
    "log_amount_received",
    "sender_transaction_count",
    "sender_unique_receivers",
    "receiver_transaction_count",
    "receiver_unique_senders",
    "sender_1h_frequency",
    "sender_1h_total_paid",
    "transaction_hour",
    "transaction_day",
    "transaction_weekday"
]

categorical_features = [
    "Receiving Currency",
    "Payment Format"
]

features = numeric_features + categorical_features

target = "Is Laundering"

In [63]:
X_train = df_main_features.loc[train, features]
X_test = df_main_features.loc[test, features]

y_train = df_main_features.loc[train, target]
y_test = df_main_features.loc[test, target]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("Training positives:", y_train.sum())
print("Testing positives:", y_test.sum())

X_train: (4214445, 12)
X_test: (862792, 12)
Training positives: 3566
Testing positives: 956


***XGBoost***

In [74]:
# Preprocess categorical features
# One-Hot Encoding: Convert categorical variables into numerical binary features.
# StandardScaler is not required for tree-based models such as XGBoost.
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

# Calculate class imbalance ratio
# Class imbalance adjustment: Calculate the ratio of negative samples to positive samples.
scale_pos_weight = (
    (y_train == 0).sum() / (y_train == 1).sum()
)

# Build XGBoost pipeline
xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="aucpr",
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

Given the severe class imbalance in our AML dataset, I used scale_pos_weight in XGBoost to increase the relative importance of positive transactions during training. This helps the model focus more on identifying minority-class transactions rather than being dominated by the majority class. I initialized the weight using the ratio of negative to positive samples and would evaluate different weights based on PR-AUC, recall, precision, and alert volume.

In [75]:
# Train the XGBoost model
xgb_model.fit(X_train, y_train)
print("XGBoost training completed.")

XGBoost training completed.


In [76]:
# Generate predicted probabilities for the positive class
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Predicted probabilities:", y_prob[:10])
print("Minimum probability:", y_prob.min())
print("Maximum probability:", y_prob.max())

Predicted probabilities: [0.23891297 0.43915504 0.2347609  0.18330042 0.06222776 0.2229948
 0.05597406 0.17308636 0.2114024  0.1311668 ]
Minimum probability: 3.3100387e-06
Maximum probability: 0.99926263


In [77]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score
)

# Evaluate predicted probabilities using a 0.5 threshold
y_pred = (y_prob >= 0.5).astype(int)

# Calculate evaluation metrics
pr_auc = average_precision_score(y_test, y_prob)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("PR-AUC:", pr_auc)
print("Precision:", precision)
print("Recall:", recall)

PR-AUC: 0.32390126604593994
Precision: 0.017401802626922837
Recall: 0.8744769874476988


In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[814631  47205]
 [   120    836]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97    861836
           1       0.02      0.87      0.03       956

    accuracy                           0.95    862792
   macro avg       0.51      0.91      0.50    862792
weighted avg       1.00      0.95      0.97    862792



***Threshold tuning***

In [79]:
y_pred = (y_prob >= 0.5).astype(int)

In [82]:
# Define candidate classification thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

results = []

for threshold in thresholds:

    # Convert predicted probabilities into binary predictions
    y_pred_threshold = (y_prob >= threshold).astype(int)

    # Calculate confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred_threshold
    ).ravel()

    # Calculate evaluation metrics
    precision = precision_score(
        y_test, y_pred_threshold, zero_division=0
    )

    recall = recall_score(
        y_test, y_pred_threshold, zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred_threshold, zero_division=0
    )

    # Calculate alert volume and false positive rate
    alert_volume = tp + fp
    false_positive_rate = fp / (fp + tn)

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "True Positives": tp,
        "False Positives": fp,
        "False Negatives": fn,
        "Alert Volume": alert_volume,
        "Alert Rate": alert_volume / len(y_test),
        "False Positive Rate": false_positive_rate
    })

# Create threshold comparison table
threshold_results = pd.DataFrame(results)

display(threshold_results.round(4))

,Threshold,Precision,Recall,F1,True Positives,False Positives,False Negatives,Alert Volume,Alert Rate,False Positive Rate
0,0.1,0.0081,0.9561,0.0161,914,111573,42,112487,0.1304,0.1295
1,0.2,0.0100,0.9257,0.0198,885,87529,71,88414,0.1025,0.1016
2,0.3,0.0119,0.9048,0.0236,865,71554,91,72419,0.0839,0.0830
3,0.4,0.0143,0.8881,0.0281,849,58571,107,59420,0.0689,0.0680
4,0.5,0.0174,0.8745,0.0341,836,47205,120,48041,0.0557,0.0548
5,0.6,0.0216,0.8588,0.0421,821,37260,135,38081,0.0441,0.0432
6,0.7,0.0259,0.8400,0.0502,803,30243,153,31046,0.0360,0.0351
7,0.8,0.0325,0.8243,0.0625,788,23469,168,24257,0.0281,0.0272
8,0.9,0.0445,0.7845,0.0843,750,16094,206,16844,0.0195,0.0187


I performed threshold tuning to evaluate the trade-off between detection performance and operational efficiency.

I compared Precision, Recall, False Positive Rate, and Alert Volume across different thresholds.

We know a lower threshold increased recall but also generated more false positives and a higher alert volume. A higher threshold reduced the alert volume but increased the risk of missing suspicious transactions.

Therefore, I would select the threshold based on **predefined business KPIs**, such as a minimum recall target, an acceptable false positive rate, and the investigation team's capacity.

***Feature importance analysis***

In [ ]:
# Get the fitted preprocessing and XGBoost model
preprocessor = xgb_model.named_steps["preprocessor"]
classifier = xgb_model.named_steps["classifier"]

# Get feature names after one-hot encoding
feature_names = preprocessor.get_feature_names_out()

# Extract feature importance scores
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": classifier.feature_importances_
})

# Sort features by importance
importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Display the top 20 features
display(importance_df.head(20).round(4))

,Feature,Importance
0,cat__Payment Format_ACH,0.6114
1,cat__Payment Format_Reinvestment,0.0840
2,remainder__sender_1h_frequency,0.0441
3,cat__Payment Format_Credit Card,0.0430
4,cat__Payment Format_Cheque,0.0290
5,remainder__receiver_unique_senders,0.0218
6,cat__Payment Format_Bitcoin,0.0206
7,remainder__sender_unique_receivers,0.0189
8,cat__Receiving Currency_Saudi Riyal,0.0183
9,cat__Payment Format_Cash,0.0143


The feature importance analysis showed that Payment Format, particularly ACH, was the dominant predictor in my model, accounting for approximately 61% of the total feature importance.

This was consistent with my exploratory data analysis, which showed a strong association between ACH transactions and positive labels.

However, I would be cautious about interpreting this result because it could reflect dataset-specific labeling patterns rather than genuine AML risk patterns. I would further investigate the model's sensitivity to this feature and evaluate its performance without Payment Format.

***Feature sensitivity analysis***

In [86]:
# Remove Payment Format from categorical features
categorical_features_no_payment = [
    "Receiving Currency"
]

# Keep all numeric features unchanged
features_no_payment = (
    numeric_features + categorical_features_no_payment
)

# Create training and testing datasets
X_train_no_payment = df_main_features.loc[
    train, features_no_payment
]

X_test_no_payment = df_main_features.loc[
    test, features_no_payment
]

print("Training shape:", X_train_no_payment.shape)
print("Testing shape:", X_test_no_payment.shape)

Training shape: (4214445, 11)
Testing shape: (862792, 11)


In [87]:
# Preprocess categorical features
preprocessor_no_payment = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features_no_payment
        )
    ],
    remainder="passthrough"
)

# Create XGBoost model without Payment Format
xgb_model_no_payment = Pipeline(
    steps=[
        ("preprocessor", preprocessor_no_payment),
        ("classifier", XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale_pos_weight,
            eval_metric="aucpr",
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train the model
xgb_model_no_payment.fit(
    X_train_no_payment,
    y_train
)

print("Model training completed.")

Model training completed.


In [88]:

# Predict probabilities
y_prob_no_payment = xgb_model_no_payment.predict_proba(
    X_test_no_payment
)[:, 1]

# Use the same threshold of 0.5
threshold = 0.5

y_pred_no_payment = (
    y_prob_no_payment >= threshold
).astype(int)

# Calculate evaluation metrics
pr_auc_no_payment = average_precision_score(
    y_test,
    y_prob_no_payment
)

precision_no_payment = precision_score(
    y_test,
    y_pred_no_payment,
    zero_division=0
)

recall_no_payment = recall_score(
    y_test,
    y_pred_no_payment,
    zero_division=0
)

f1_no_payment = f1_score(
    y_test,
    y_pred_no_payment,
    zero_division=0
)

# Calculate confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_test,
    y_pred_no_payment
).ravel()

# Calculate operational metrics
alert_volume_no_payment = tp + fp
alert_rate_no_payment = alert_volume_no_payment / len(y_test)
fpr_no_payment = fp / (fp + tn)

print("PR-AUC:", round(pr_auc_no_payment, 4))
print("Precision:", round(precision_no_payment, 4))
print("Recall:", round(recall_no_payment, 4))
print("F1:", round(f1_no_payment, 4))
print("False Positive Rate:", round(fpr_no_payment, 4))
print("Alert Volume:", alert_volume_no_payment)
print("Alert Rate:", round(alert_rate_no_payment, 4))
print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)

PR-AUC: 0.2015
Precision: 0.0097
Recall: 0.8755
F1: 0.0192
False Positive Rate: 0.0991
Alert Volume: 86219
Alert Rate: 0.0999
True Positives: 837
False Positives: 85382
False Negatives: 119


In [89]:
comparison = pd.DataFrame({
    "Metric": [
        "PR-AUC",
        "Precision",
        "Recall",
        "F1",
        "False Positive Rate",
        "Alert Volume"
    ],
    "With Payment Format": [
        average_precision_score(y_test, y_prob),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        47205 / (47205 + 814631),
        48041
    ],
    "Without Payment Format": [
        pr_auc_no_payment,
        precision_no_payment,
        recall_no_payment,
        f1_no_payment,
        fpr_no_payment,
        alert_volume_no_payment
    ]
})

display(comparison.round(4))

,Metric,With Payment Format,Without Payment Format
0,PR-AUC,0.3239,0.2015
1,Precision,0.0174,0.0097
2,Recall,0.8745,0.8755
3,F1,0.0341,0.0192
4,False Positive Rate,0.0548,0.0991
5,Alert Volume,48041.0000,86219.0000


I conducted a feature sensitivity analysis by removing Payment Format and retraining the XGBoost model.

Without this feature, PR-AUC decreased from 0.324 to 0.202, while recall remained almost unchanged.

However, alert volume increased from approximately 48,000 to 86,000 at the same threshold of 0.5.

This suggests that Payment Format provides significant predictive information in this dataset. However, I would investigate whether this relationship reflects genuine transaction risk or dataset-specific labeling patterns before relying on it in a production AML system.

In [ ]:
threshold_results = pd.DataFrame({
    "Threshold": [0.1, 0.2, 0.3, 0.4, 0.5,
                  0.6, 0.7, 0.8, 0.9],
    "Precision": [0.0081, 0.0100, 0.0119, 0.0143, 0.0174,
                  0.0216, 0.0259, 0.0325, 0.0445],
    "Recall": [0.9561, 0.9257, 0.9048, 0.8881, 0.8745,
               0.8588, 0.8400, 0.8243, 0.7845],
    "Alert Volume": [112487, 88414, 72419, 59420, 48041,
                     38081, 31046, 24257, 16844]
})

# Define hypothetical business constraints
max_alerts = 25000
min_recall = 0.80

# Filter thresholds that satisfy both constraints
eligible_thresholds = threshold_results[
    (threshold_results["Alert Volume"] <= max_alerts) &
    (threshold_results["Recall"] >= min_recall)
]

display(eligible_thresholds)

,Threshold,Precision,Recall,Alert Volume
7,0.8,0.0325,0.8243,24257


A **hypothetical alert investigation capacity of 25,000 alerts** was used to simulate operational constraints. A **minimum recall target of 80%** was also applied.

A threshold of 0.8 was identified as a suitable candidate because it generated 24,257 alerts while maintaining a recall of 82.43%. Higher thresholds reduced alert volume further but caused recall to fall below the assumed detection target.

This analysis illustrates how model thresholds can be selected by balancing detection performance with operational investigation capacity.

#### **Model Development, Evaluation, and Threshold Selection Report**

**1. Project Objective**

The objective of this project was to develop and evaluate a machine learning model for identifying transactions labeled as potentially related to money laundering. Beyond predictive performance, I focused on understanding transaction behavior, engineering historical behavioral features, and evaluating how classification thresholds affect detection performance and investigation workload.

Dataset: IBM Transactions for Anti-Money Laundering (AML), HI-Small
Primary analysis period: September 1–10, 2022
Model: XGBoost Classifier

**2. Data Quality Assessment and Exploratory Data Analysis**

**2.1 Data Quality and Data Suitability**

I first assessed data quality, including missing values and duplicate records, to determine which data was suitable for analysis. I also reviewed transaction volume and positive-label distributions over time to select an appropriate period for modeling.

The dataset showed substantial changes in transaction volume and positive-label rates across different periods. I therefore selected September 1–10 as my primary modeling period rather than combining periods with significantly different distributions.

**2.2 Exploratory Data Analysis**

I explored the original transaction variables and their relationships with the dataset's positive labels, focusing on:

Transaction amounts: Examined distributions and identified substantial right skewness and extreme values.

Payment formats: Observed differences in positive-label rates across transaction formats. ACH transactions accounted for approximately 84.65% of positive labels in the selected period.

Currencies: Examined currency patterns and relationships with transaction amounts and labels.

Account activity: Investigated transaction concentration and historical activity across sender accounts.

Temporal patterns: Examined transaction frequency and activity over different time periods.

The analysis suggested that individual transaction attributes alone might not fully capture suspicious behavior. This motivated the development of historical behavioral features.

**3. Feature Engineering**

My main objective in feature engineering was to represent transaction behavior mathematically, going beyond the characteristics of an individual transaction.

**3.1 Historical Account Behavior**

I developed features to capture account activity and transaction relationships:

sender_transaction_count: Historical transaction count for the sender.

sender_unique_receivers: Number of distinct receivers previously associated with the sender.

receiver_transaction_count: Historical transaction count for the receiver.

receiver_unique_senders: Number of distinct senders previously associated with the receiver.

**3.2 Short-Term Transaction Behavior**

I also engineered rolling one-hour features to capture changes in transaction activity:

sender_1h_frequency: Number of transactions initiated by the sender during the preceding hour.

sender_1h_total_paid: Total payment amount initiated by the sender during the preceding hour.

These features were calculated using prior transactions only, excluding the current transaction and transactions with the same timestamp, to avoid look-ahead bias.

**3.3 Additional Feature Preparation**

I applied a logarithmic transformation to transaction amounts to reduce the impact of extreme values.

I also extracted temporal features, including transaction hour, day, and weekday.

For categorical variables, such as Payment Format and Receiving Currency, I used One-Hot Encoding to create binary indicator features.

The final feature set included 10 numerical predictors and 2 categorical predictors. Raw account identifiers were excluded from the model.

**4. Model Development**

I selected XGBoost as the classification model and used a chronological training and testing split to better reflect a time-based transaction monitoring scenario.

| Dataset | Transactions | Positive Labels |
|---|---:|---:|
| Training | 4,214,445 | 3,566 |
| Testing | 862,792 | 956 |

The positive class represented a very small proportion of the transactions, creating a severe class imbalance.

To address this, I used the scale_pos_weight parameter in XGBoost. The weight was initialized using the ratio of negative to positive observations in the training data.

I also used One-Hot Encoding for categorical variables and configured the encoder to handle previously unseen categories.

**5. Model Evaluation and Diagnostics**

Given the severe class imbalance, I focused on PR-AUC, precision, recall, and false positive rate rather than relying on accuracy alone.

At the initial classification threshold of 0.5, the model achieved the following results:

| Metric | Result |
|---|---:|
| PR-AUC | 0.3239 |
| Precision | 1.74% |
| Recall | 87.45% |
| False Positive Rate | 5.48% |
| Alert Volume | 48,041 |

The confusion matrix was:

| | Predicted Negative | Predicted Positive |
|---|---:|---:|
| Actual Negative | 814,631 | 47,205 |
| Actual Positive | 120 | 836 |

The model identified approximately 87.45% of the positive labels, but it also generated a substantial number of false positives. This highlighted the need to evaluate different classification thresholds to manage investigation workload.

** Feature Importance Analysis**

I examined XGBoost feature importance to understand which predictors contributed most to the model.

Payment Format, particularly the ACH category, had the largest feature importance. This was consistent with the patterns observed during EDA.

However, feature importance indicates predictive contribution within the fitted model; it does not establish causation or prove that a particular transaction format represents higher real-world money laundering risk.

I also conducted a feature sensitivity experiment by removing Payment Format and retraining the model. PR-AUC decreased from 0.3239 to 0.2015, while recall remained almost unchanged. This demonstrated the model's substantial reliance on Payment Format in this dataset.

**6. Threshold Sensitivity Analysis**

Because the default classification threshold of 0.5 generated a large number of alerts, I evaluated different thresholds to understand the trade-off between detection performance and investigation workload. I evaluated classification thresholds ranging from 0.1 to 0.9 to understand their impact on detection performance and alert volume.

| Threshold | Precision | Recall | Alert Volume | FPR |
|---:|---:|---:|---:|---:|
| 0.1 | 0.81% | 95.61% | 112,487 | 12.95% |
| 0.2 | 1.00% | 92.57% | 88,414 | 10.16% |
| 0.3 | 1.19% | 90.48% | 72,419 | 8.30% |
| 0.4 | 1.43% | 88.81% | 59,420 | 6.80% |
| 0.5 | 1.74% | 87.45% | 48,041 | 5.48% |
| 0.6 | 2.16% | 85.88% | 38,081 | 4.32% |
| 0.7 | 2.59% | 84.00% | 31,046 | 3.51% |
| 0.8 | 3.25% | 82.43% | 24,257 | 2.72% |
| 0.9 | 4.45% | 78.45% | 16,844 | 1.87% |

The results showed a trade-off between detection performance and investigation workload. As the threshold increased, precision improved and alert volume decreased, while recall also declined.

**7. Threshold Selection Rationale**

To connect model evaluation with operational considerations, I introduced two hypothetical business constraints:

- Maximum investigation capacity: 25,000 alerts.
- Minimum recall requirement: 80%.

Based on these assumptions, I compared the candidate thresholds.

At a threshold of 0.5, the model generated 48,041 alerts, exceeding the assumed investigation capacity.

At a threshold of 0.8, the alert volume decreased to 24,257, while recall remained at 82.43%. Both hypothetical requirements were satisfied.

At a threshold of 0.9, alert volume decreased further to 16,844, but recall fell to 78.45%, below the assumed minimum requirement.

Therefore, 0.8 was selected as a candidate threshold under these hypothetical operational constraints.

**8. Limitations and Considerations**

Several limitations should be considered when interpreting the results:

**Synthetic dataset:**
The positive labels are dataset-provided labels and should not be interpreted as independently verified money laundering cases.

**Temporal distribution shifts:**
The dataset exhibited substantial changes in transaction volume and positive-label rates across different periods.

**Feature dependence:**
Payment Format had a dominant influence on model predictions, potentially reflecting dataset-specific label patterns.

**Currency comparability:**
The rolling one-hour cumulative payment amount was calculated using raw nominal amounts without foreign exchange normalization.

**Threshold assumptions:**
The investigation capacity of 25,000 alerts and minimum recall target of 80% were hypothetical assumptions, not actual operational requirements.

**Threshold validation:**
The threshold comparison and candidate selection were exploratory analyses conducted on the same test set. The selected threshold would require further validation on an independent dataset or subsequent time period before operational deployment.

**9. Conclusion**

This project covered the end-to-end analytical process of AML transaction monitoring, from data quality assessment and exploratory analysis to behavioral feature engineering, model development, model evaluation, and threshold selection.

My main focus was to combine quantitative modeling with practical decision-making. By translating historical transaction behavior into measurable features and evaluating threshold trade-offs under hypothetical operational constraints, I demonstrated how statistical analysis can support both risk detection and investigation resource management.